In [ ]:
# =============================================================================
# COLAB BOOTSTRAP — RUN THIS CELL FIRST.
# Lets a Google Colab user upload the data files from their computer and saves
# them into the notebook's working directory, so the relative paths used by
# every cell below ('result.csv', 'stability_results.csv') resolve unchanged.
#   * result.csv          — required by the BO-data cells (LHS, heatmap,
#                           convergence, valid-designs scatter, iteration plots).
#                           Take it from results_archive/best_run_109/result.csv
#                           (or 'simulation csv/result.csv').
#   * stability_results.csv — ONLY needed by the wavelength-deviation cell.
#                           From results_archive/wavelength_109/.
# Outside Colab this is a no-op as long as the files already sit next to the
# notebook (or, for stability_results.csv, in the archive paths that cell checks).
# =============================================================================
import os

try:
    from google.colab import files  # available only inside Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REQUIRED = "result.csv"
OPTIONAL = "stability_results.csv"

if IN_COLAB:
    missing = [f for f in (REQUIRED, OPTIONAL) if not os.path.exists(f)]
    if missing:
        print("Upload from your computer:", ", ".join(missing))
        print("(result.csv is required; stability_results.csv is only for the "
              "wavelength plot — press Cancel to skip it.)")
        uploaded = files.upload()  # opens the file picker; saves into the cwd
        for name in uploaded:
            print("  saved ->", os.path.abspath(name))
    else:
        print("Both data files already present — nothing to upload.")
else:
    print("Not running in Colab — using local files in:", os.getcwd())

# Expose the resolved absolute paths globally for the rest of the notebook.
RESULT_CSV = os.path.abspath(REQUIRED) if os.path.exists(REQUIRED) else None
STABILITY_CSV = os.path.abspath(OPTIONAL) if os.path.exists(OPTIONAL) else None
print("RESULT_CSV    =", RESULT_CSV)
print("STABILITY_CSV =", STABILITY_CSV)
if RESULT_CSV is None:
    print("WARNING: result.csv not found — the BO-data cells will fail until it is present.")


In [ ]:
# =============================================================================
# DATA SOURCE: RESULT_CSV  — resolved by the Colab-bootstrap cell at the top.
#   Points to the result.csv you upload in Colab (or the local file otherwise);
#   original source: results_archive/best_run_109/result.csv (full BO run,
#   150 sims, sim_id 1..150).
# This cell: LHS scatter (V_pi*L vs loss, valid/invalid), uses sim_id < 50.
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io

# 1. Load and clean the CSV data
with open(RESULT_CSV, 'rb') as f:
    raw_data = f.read().decode('utf-8', errors='ignore')

# Find the start of the actual CSV header
csv_start = raw_data.find('sim_id')
if csv_start == -1:
    csv_start = raw_data.find(r'sim\_id') # Changed to raw string to fix SyntaxWarning
clean_csv = raw_data[csv_start:]

df = pd.read_csv(io.StringIO(clean_csv))
df.columns = [c.replace('\\', '') for c in df.columns]

# Force columns to numeric (fixes the string/integer comparison error)
df['sim_id'] = pd.to_numeric(df['sim_id'], errors='coerce')
df['v_pi_l_Vmm'] = pd.to_numeric(df['v_pi_l_Vmm'], errors='coerce')
df['loss_at_v_pi_dB_per_cm'] = pd.to_numeric(df['loss_at_v_pi_dB_per_cm'], errors='coerce')
df['max_dphi_rad'] = pd.to_numeric(df['max_dphi_rad'], errors='coerce')

# Filter for LHS samples only (sim_id < 50)
df_lhs = df[df['sim_id'] < 50].copy()

# 2. Define Validity Conditions
alpha_target = 20.0
alpha_max = alpha_target * 3  # 60.0 dB/cm

# Valid: Reached Pi AND Loss is within bounds
valid_mask = (df_lhs['max_dphi_rad'] >= np.pi) & (df_lhs['loss_at_v_pi_dB_per_cm'] <= alpha_max)

valid_df = df_lhs[valid_mask]
invalid_df = df_lhs[~valid_mask]

# Use original absorption coefficient for plotting and set y-axis to log scale
valid_x = valid_df['v_pi_l_Vmm']
valid_y = valid_df['loss_at_v_pi_dB_per_cm']

invalid_x = invalid_df['v_pi_l_Vmm']
invalid_y = invalid_df['loss_at_v_pi_dB_per_cm']

# 3. Plotting the data
plt.figure(figsize=(9, 6))

# Plot Valid points
plt.scatter(valid_x, valid_y,
            color='#5BA4DA', edgecolor='#4D4D4D', s=70, marker='o', alpha=0.8,
            label=r'Valid (Reached $\pi$ & $\alpha \leq \alpha_{max}$)')

# Plot Invalid points
plt.scatter(invalid_x, invalid_y,
            color='#F15854', edgecolor='#4D4D4D', s=70, marker='X', alpha=0.9,
            label=r'Invalid (Did not reach $\pi$ or $\alpha > \alpha_{max}$)')

# Formatting to match the provided image style
plt.xlabel(r'Modulation Efficiency (V$\cdot$mm)', fontsize=12, fontweight='bold')
plt.ylabel(r'Absorption Coeff (dB/cm)', fontsize=12, fontweight='bold') # Changed label
plt.grid(True, linestyle='--', alpha=0.5)

plt.xlim(left=0)
plt.ylim(bottom=0.1, top=100000) # Adjusted for log scale
plt.yscale('log') # Set y-axis to log scale

# Styling the legend
plt.legend(loc='center right', frameon=True, shadow=True, edgecolor='gray')
plt.tight_layout()

plt.show()

In [ ]:
# =============================================================================
# DATA SOURCE: RESULT_CSV  — resolved by the Colab-bootstrap cell at the top.
#   Points to the result.csv you upload in Colab (or the local file otherwise);
#   original source: results_archive/best_run_109/result.csv (full BO run,
#   150 sims, sim_id 1..150).
# This cell: Spearman correlation heatmap of params vs metrics, uses sim_id < 50.
# =============================================================================
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import io

# 1. Load the result data
# Bypassing the binary header artifacts in the raw CSV output
with open(RESULT_CSV, 'rb') as f:
    raw_data = f.read().decode('utf-8', errors='ignore')

csv_start = raw_data.find('sim_id')
if csv_start == -1:
    csv_start = raw_data.find(r'sim\_id') # Changed to raw string to fix SyntaxWarning
clean_csv = raw_data[csv_start:]

df = pd.read_csv(io.StringIO(clean_csv))
df.columns = [c.replace('\\', '') for c in df.columns]

# Filter for sim_id < 50
df = df[df['sim_id'] < 50]

# 2. Select relevant parameters and performance metrics
features = [
    'w_r', 'h_si', 'doping', 'S', 'length',
    'v_pi_l_Vmm', 'loss_at_v_pi_dB_per_cm', 'cost'
]
cols_to_use = [c for c in features if c in df.columns]
corr_df = df[cols_to_use].dropna()

# 3. Compute the Spearman rank correlation
corr_matrix = corr_df.corr(method='spearman')

# 4. Generate the Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f', linewidths=0.5)
plt.tight_layout()

plt.show()

In [ ]:
# =============================================================================
# DATA SOURCE: none — this 3D cost surface is computed analytically from the
#   cost function (meshgrid), so no result file needs to be loaded.
# =============================================================================
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. System Constraints & Targets
# ==========================================
VpiL_target = 1
alpha_target = 20.0
alpha_max = alpha_target * 3.0  # 60.0 dB/cm
V_MAX = 2.5
L_MAX = 1.0

# FOM Weights
w_loss = 0.9
w_VpiL = 1 - w_loss

# ==========================================
# 2. Penalty Configuration (from config.py)
# ==========================================
# Calculate theoretical worst-case valid cost to establish C_Base
WORST_CASE_VPIL = V_MAX * L_MAX
MAX_VALID_COST = (w_loss * (alpha_max / alpha_target)**2) + (w_VpiL * (WORST_CASE_VPIL / VpiL_target)**2)

# C_Base dynamically prevents cost inversion between Branch A and Branch B
C_Base = MAX_VALID_COST + 2.0
BETA_ELEC = (9.0 * C_Base) / (np.pi**2)
BETA_OPT = C_Base / 100.0

# ==========================================
# 3. Cost Function Logic (from cost.py)
# ==========================================
def calculate_cost(alpha, v_pi_l, max_dphi, targets, weights):
    """
    Calculates the multi-objective cost using a dual-condition piecewise penalty.
    """
    current_alpha_max = 3.0 * targets['loss']

    # Dual-condition validity check
    is_electrically_valid = max_dphi >= np.pi
    is_optically_valid = alpha <= current_alpha_max

    # Branch A: Valid Device
    if is_electrically_valid and is_optically_valid:
        norm_loss = alpha / targets['loss']
        norm_vpil = v_pi_l / targets['vpil']
        cost = (weights['loss'] * (norm_loss**2)) + (weights['vpil'] * (norm_vpil**2))
        return cost

    # Branch B: Invalid Device (Penalty Region)
    else:
        # Cap alpha to prevent Lumerical numerical anomalies from skewing the GP
        capped_alpha = min(alpha, 1000.0)

        elec_penalty = BETA_ELEC * (max(0, np.pi - max_dphi)**2)
        opt_penalty = BETA_OPT * max(0, (capped_alpha - current_alpha_max)**2)

        return C_Base + weights['vpil']*elec_penalty + weights['loss']*opt_penalty

# ==========================================
# 4. Generate Meshgrid and Calculate Costs
# ==========================================
VpiL_vals = np.linspace(0.1, 3.5, 100)
alpha_vals = np.linspace(0.1, 100.0, 100)
V, A = np.meshgrid(VpiL_vals, alpha_vals)
Cost = np.zeros_like(V)

targets = {'loss': alpha_target, 'vpil': VpiL_target}
weights = {'loss': w_loss, 'vpil': w_VpiL}

# We take a 3D slice of the 4D space by assuming the device successfully
# achieved a pi phase shift. This cleanly reveals the Branch A optimization
# basin and the Branch B optical penalty wall without falsifying physics.
fixed_max_dphi = np.pi

# Compute the cost over the meshgrid
for i in range(V.shape[0]):
    for j in range(V.shape[1]):
        raw_cost = calculate_cost(A[i,j], V[i,j], fixed_max_dphi, targets, weights)
        Cost[i,j] = np.log10(raw_cost)

# ==========================================
# 5. Generate the 3D Plot
# ==========================================
fig = plt.figure(figsize=(10, 12))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(V, A, Cost, cmap='inferno', edgecolor='none', alpha=0.85)

# --- ADDING THE TARGET REGION BOUNDING BOX ---
z_max_target = 0.0
z_min_target = np.min(Cost)

x_box = [0.1, VpiL_target, VpiL_target, 0.1, 0.1]
y_box = [0.1, 0.1, alpha_target, alpha_target, 0.1]

# Top face of the target box
z_box_top = [z_max_target] * 5
ax.plot(x_box, y_box, z_box_top, color='red', linewidth=2.0, linestyle='--')

# Bottom face of the target box
z_box_bot = [z_min_target] * 5
ax.plot(x_box, y_box, z_box_bot, color='red', linewidth=2.0, linestyle='--', label=r'Target Region')

# Vertical drop lines to make it a 3D volume
for x, y in zip(x_box[:-1], y_box[:-1]):
    ax.plot([x, x], [y, y], [z_min_target, z_max_target], color='red', linestyle='--', linewidth=2.0)

# Labels and formatting
ax.set_xlabel(r'Modulation Efficiency (V·mm)', fontsize=12, fontweight='bold', labelpad=10)
ax.set_ylabel(r'Absorption Coeff (dB/cm)', fontsize=12, fontweight='bold', labelpad=10)
ax.set_zlabel(r'$log_{10}(Cost)$', fontsize=12, fontweight='bold', labelpad=10)

ax.view_init(elev=25, azim=-75)
# fig.colorbar(surf, shrink=0.5, aspect=10, label=r'$log_{10}(Cost)$')
ax.legend(loc='center left')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# DATA SOURCE: RESULT_CSV  — resolved by the Colab-bootstrap cell at the top.
#   Points to the result.csv you upload in Colab (or the local file otherwise);
#   original source: results_archive/best_run_109/result.csv (full BO run,
#   150 sims, sim_id 1..150).
# This cell: cost-convergence plot (all sims); saves cost_convergence_slide.png.
# =============================================================================
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import io

with open(RESULT_CSV, 'rb') as f:
    raw_data = f.read().decode('utf-8', errors='ignore')

csv_start = raw_data.find('sim_id')
if csv_start == -1:
    csv_start = raw_data.find('sim\\_id')
clean_csv = raw_data[csv_start:]

df = pd.read_csv(io.StringIO(clean_csv))
df.columns = [c.replace('\\', '') for c in df.columns]
df['sim_id'] = pd.to_numeric(df['sim_id'], errors='coerce')
df['cost'] = pd.to_numeric(df['cost'], errors='coerce')
df = df.dropna(subset=['sim_id', 'cost']).sort_values('sim_id')

iterations = df['sim_id'].values
costs = df['cost'].values

best_costs = np.minimum.accumulate(costs)

improvement_indices = [0]
for i in range(1, len(best_costs)):
    if best_costs[i] < best_costs[i-1]:
        improvement_indices.append(i)

# Compact, slide-friendly figure (narrower than the original 12x7)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(iterations, costs, color='gray', alpha=0.3, s=16, label='Iteration Cost Values')
ax.step(iterations, best_costs, where='post', color='#1b3b6f', linewidth=2.2, label='Best Found Cost')

ax.set_yscale('log')
ax.set_ylim(bottom=0.1, top=1500)  # headroom on top so the labels stay inside the axes

# x in data coords, y in axes fraction -> labels sit just below the top edge (never above the plot)
trans = ax.get_xaxis_transform()
for idx in improvement_indices:
    iter_val = iterations[idx]
    ax.axvline(x=iter_val, linestyle=':', color='#7209b7', alpha=0.7, linewidth=1.3)
    ax.text(iter_val, 0.985, f'{int(iter_val)}', transform=trans, color='#7209b7',
            rotation=35, va='top', ha='center', fontsize=8, fontweight='bold')

ax.axvline(x=50, linestyle='-', color='red', alpha=0.5, linewidth=1.5)
ax.text(50, 0.985, 'LHS End', transform=trans, color='red',
        rotation=75, va='top', ha='center', fontsize=9, fontweight='bold')

ax.set_xlabel('Simulation Iteration', fontsize=11, fontweight='bold')
ax.set_ylabel('$Log_{10}(Cost)$', fontsize=11, fontweight='bold')
ax.grid(True, which="both", ls="--", alpha=0.2)
ax.legend(loc='center right', fontsize=9)

fig.tight_layout()
fig.savefig('cost_convergence_slide.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# DATA SOURCE: RESULT_CSV  — resolved by the Colab-bootstrap cell at the top.
#   Points to the result.csv you upload in Colab (or the local file otherwise);
#   original source: results_archive/best_run_109/result.csv (full BO run,
#   150 sims, sim_id 1..150).
# This cell: valid-designs scatter with the global-optimum star + target box.
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import io

# 1. טעינה וניקוי הנתונים
with open(RESULT_CSV, 'rb') as f:
    raw_data = f.read().decode('utf-8', errors='ignore')

csv_start = raw_data.find('sim_id')
if csv_start == -1:
    csv_start = raw_data.find(r'sim\_id')
clean_csv = raw_data[csv_start:]

df = pd.read_csv(io.StringIO(clean_csv))
df.columns = [c.replace('\\', '') for c in df.columns]

# המרה לפורמט נומרי
cols_to_convert = ['sim_id', 'v_pi_l_Vmm', 'loss_at_v_pi_dB_per_cm', 'max_dphi_rad', 'cost']
for c in cols_to_convert:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df = df.dropna(subset=['sim_id', 'cost'])

# 2. סינון קונפיגורציות
alpha_max = 60.0
# אנו מתמקדים בגרף זה רק בנקודות שצלחו את המגבלות החשמליות והאופטיות כדי להציג את אגן האופטימיזציה
valid_df = df

# מציאת נקודת האופטימום הגלובלי (המחיר המינימלי ביותר)
best_idx = valid_df['cost'].idxmin()
best_design = valid_df.loc[best_idx]

# 3. יצירת הגרף
plt.figure(figsize=(10, 7))

# ציור כלל הנקודות התקינות (חזית התכנון)
plt.scatter(valid_df['v_pi_l_Vmm'], valid_df['loss_at_v_pi_dB_per_cm'],
            color='#1b3b6f', alpha=0.6, s=50, edgecolor='black', linewidth=0.5,
            label='Valid Evaluated Designs')

# סימון נקודת האופטימום הגלובלי בכוכב זהב ענק
# Fixed the format specifier from :.f0 to :.0f
plt.scatter(best_design['v_pi_l_Vmm'], best_design['loss_at_v_pi_dB_per_cm'],
            color='#f4a300', marker='*', s=450, edgecolor='black', linewidth=0.5, zorder=5,
            label=f'Global Optimum (Sim ID: {best_design["sim_id"]:.0f})')

# הוספת תיבת היעד (Target Region)
ax = plt.gca()
target_box = patches.Rectangle((0, 0), 0.8, 20.0, linewidth=2, edgecolor='#9bbb59', facecolor='lightgreen', alpha=0.3, zorder=1)
ax.add_patch(target_box)
plt.plot([], [], color='#9bbb59', linewidth=10, alpha=0.3, label=r'Target Region ($V_\pi L \leq 0.8$, $\alpha \leq 20$)')

# עיצוב הצירים והגרף
plt.xlim(left=0.0, right=1.1)
plt.ylim(bottom=0.0, top=65.0)

plt.xlabel(r'Modulation Efficiency (V·mm)', fontsize=12, fontweight='bold')
plt.ylabel(r'Absorption Coeff (dB/cm)', fontsize=12, fontweight='bold')

plt.grid(True, linestyle='--', alpha=0.25)
plt.legend(loc='upper right', frameon=True, shadow=True)

plt.tight_layout()
plt.show()

# הדפסת נתוני האופטימום לקונסול
print("--- THE GLOBAL OPTIMUM PARAMETERS ---")
print(f"sim_id: {best_design['sim_id']}")
print(f"Wr: {best_design['w_r']*1e9:.1f} nm")
print(f"h_si: {best_design['h_si']*1e9:.1f} nm")
print(f"Doping: {best_design['doping']:.2e} cm^-3")
print(f"S (Offset): {best_design['S']*1e9:.1f} nm")
print(f"Length: {best_design['length']*1e3:.3f} mm")
print(f"Vpi*L: {best_design['v_pi_l_Vmm']:.3f} V*mm")
print(f"Loss: {best_design['loss_at_v_pi_dB_per_cm']:.2f} dB/cm")

In [ ]:
# =============================================================================
# DATA SOURCE: result.csv  — the full BO run (150 sims, sim_id 1..150).
#   Reuses the `df` loaded in the previous cell (the valid-designs scatter); run that cell first.
# This cell: V_pi*L and loss vs simulation iteration (LHS-end & best-design marks).
# =============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

# Figure 1: V_pi*L (V·mm) vs Simulation ID
plt.figure(figsize=(12, 5))
sns.lineplot(data=df, x='sim_id', y='v_pi_l_Vmm', marker='o', color='blue')
plt.ylabel(r'Modulation Efficiency (V·mm)',fontsize=12, fontweight='bold')
plt.xlabel('Simulation Itteration', fontsize=12, fontweight='bold')
# plt.ylim(0,1)
plt.grid(True, linestyle='--', alpha=0.6)
plt.axvline(x=50, linestyle='-', color='red', alpha=0.5)
plt.text(50, plt.gca().get_ylim()[1], f'LHS End',
             color='red', rotation=90, verticalalignment='top', fontsize=9, fontweight='bold')
plt.axvline(x=109, linestyle='-', color='g', alpha=0.5)
plt.text(109, plt.gca().get_ylim()[1], f'Best Design',
             color='g', rotation=90, verticalalignment='top', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

# Figure 2: Loss at V_pi (dB/cm) vs Simulation ID (log scale)
plt.figure(figsize=(12, 5))
sns.lineplot(data=df, x='sim_id', y=df['loss_at_v_pi_dB_per_cm'], marker='o', color='purple')
plt.yscale('log')
plt.ylabel('Absorption Coeff (dB/cm)',fontsize=12, fontweight='bold')
plt.xlabel('Simulation Itteration', fontsize=12, fontweight='bold')
# plt.ylim(10,100)
plt.grid(True, linestyle='--', alpha=0.6)
plt.axvline(x=50, linestyle='-', color='red', alpha=0.5)
plt.text(50, plt.gca().get_ylim()[1], f'LHS End',
             color='red', rotation=90, verticalalignment='top', fontsize=9, fontweight='bold')
plt.axvline(x=109, linestyle='-', color='g', alpha=0.5)
plt.text(109, plt.gca().get_ylim()[1], f'Best Design',
             color='g', rotation=90, verticalalignment='top', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# ==========================================================================
# Wavelength-stability deviation plot (full O-band sweep, sim_id 109).
# X: wavelength (nm).  Y: % deviation of V_pi and optical loss from 1310 nm.
# Source data produced by system/run_specific_wavelength.py.
# DATA SOURCE: STABILITY_CSV (from the Colab-bootstrap cell), with fallbacks to
#   stability_results.csv / results_archive/wavelength_109/ for local runs.
# ==========================================================================

# Locate the sweep results regardless of the notebook's working directory.
candidates = [
    STABILITY_CSV,                                       # from the Colab-bootstrap cell
    'stability_results.csv',
    '../results_archive/wavelength_109/stability_results.csv',
    'results_archive/wavelength_109/stability_results.csv',
]
csv_path = next((p for p in candidates if p and os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError('stability_results.csv not found in: ' + ', '.join(candidates))

df = pd.read_csv(csv_path).sort_values('lambda_nm')

ANCHOR_NM = 1310.0
anchor = df[df['lambda_nm'].round(2) == ANCHOR_NM].iloc[0]

# Geometry (incl. length L) is fixed across the sweep, so V_pi*L carries no
# information beyond V_pi -> plot V_pi, the quantity that actually varies.
dev_v = (df['v_pi_V'] / anchor['v_pi_V'] - 1.0) * 100.0
dev_a = (df['loss_at_v_pi_dB_per_cm'] / anchor['loss_at_v_pi_dB_per_cm'] - 1.0) * 100.0

# Presentation palette (matches the rest of the deck).
C_BLUE, C_PURPLE, C_GOLD, C_GREEN = '#1B3B6F', '#7209B7', '#F4A300', '#52796F'

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df['lambda_nm'], dev_v, 'o-', color=C_PURPLE, lw=2.2, ms=6,
        label=r'Drive voltage  $V_\pi$')
ax.plot(df['lambda_nm'], dev_a, 's-', color=C_GREEN, lw=2.2, ms=6,
        label=r'Optical loss  $\alpha$')

ax.axhline(0, color='#888', lw=1, zorder=0)
ax.axvline(ANCHOR_NM, color=C_GOLD, ls=':', lw=1.5)
ax.annotate(f'{ANCHOR_NM:.0f} nm', xy=(ANCHOR_NM, ax.get_ylim()[1]),
            xytext=(2, -4), textcoords='offset points',
            color=C_GOLD, fontweight='bold', va='top', fontsize=10)

ax.set_xlabel('Wavelength (nm)', fontsize=12, fontweight='bold')
ax.set_ylabel(f'Performance Deviation from {ANCHOR_NM:.0f} nm (%)',
              fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, framealpha=0.9)

fig.tight_layout()
fig.savefig('wavelength_deviation.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"V_pi  deviation: {dev_v.min():+.3f}% .. {dev_v.max():+.3f}%")
print(f"loss  deviation: {dev_a.min():+.3f}% .. {dev_a.max():+.3f}%")
